In [29]:
import pandas as pd
import numpy as np
import difflib
import requests
import time
from tqdm.notebook import tqdm

df = pd.read_csv('../data/processed/tfnsw_ev_cleaned.csv')
dc_chargers = df[df['is_dc_fast']==True].copy()
print(f"Targeting {len(dc_chargers)} DC Fast Chargers for API augmentation.")

Targeting 433 DC Fast Chargers for API augmentation.


In [33]:
import os
from dotenv import load_dotenv

# Load variables from the .env file
load_dotenv()

# Fetch the key securely
API_KEY = os.getenv("OCM_API_KEY")

if not API_KEY:
    print("Error: API Key not found. Check your .env file!")

In [35]:
ENDPOINT = "https://api.openchargemap.io/v3/poi" 
HEADERS = {"X-API-Key": API_KEY, "User-Agent": "Just-A-Mate-Fetching-EV-Stats/1.0"}

aug_data =[]

print("Starting API calls (sit tight and chill) ... it takes some time...")

for i, r in tqdm(dc_chargers.iterrows(), total=len(dc_chargers)):
    params ={
        "latitude": r['latitude'],
        "longitude": r['longitude'],
        "distance": 3.0,
        "distanceunit": "KM",
        "maxresults": 5,
        "compact": False,
        "verbose": False
    }

    try:
        response = requests.get(ENDPOINT, headers=HEADERS, params=params, timeout=10)
        response.raise_for_status()
        results = response.json()
        
        if results:
            # Default to the absolute closest charger in the radius
            best_poi = results[0] 
            match_method = 'Proximity Only (Closest)'
            
            # Finding a better match using the operator name
            for poi in results:
                raw_op = poi.get('OperatorInfo')
                ocm_operator = raw_op.get('Title', 'Unknown') if isinstance(raw_op, dict) else 'Unknown'
                tfnsw_op = str(r['operator']).lower()
                
                if tfnsw_op in ocm_operator.lower() or ocm_operator.lower() in tfnsw_op:
                    best_poi = poi
                    match_method = 'Proximity + Operator Match'
                    break  
            
            # Extract data from the POI
            final_raw_op = best_poi.get('OperatorInfo')
            final_ocm_operator = final_raw_op.get('Title', 'Unknown') if isinstance(final_raw_op, dict) else 'Unknown'
            usage_cost = best_poi.get('UsageCost', np.nan)
            connections = best_poi.get('Connections', [])
            
            plug_types = [c.get('ConnectionType', {}).get('Title') for c in connections if isinstance(c, dict) and isinstance(c.get('ConnectionType'), dict)]
            plug_types = ", ".join(filter(None, plug_types)) if plug_types else "Unknown"
            
            aug_data.append({
                'charger_id': r['charger_id'],
                'ocm_id': best_poi.get('ID'),
                'ocm_operator': final_ocm_operator,
                'usage_cost': usage_cost,
                'ocm_plug_count': len(connections),
                'ocm_plug_types': plug_types,
                'match_method': match_method
            })
                
        else:
            aug_data.append({
                'charger_id': r['charger_id'], 'ocm_id': np.nan, 'ocm_operator': 'Unknown', 
                'usage_cost': np.nan, 'ocm_plug_count': 0, 'ocm_plug_types': 'Unknown', 
                'match_method': 'No Match'
            })
            
    except Exception as e:
        print(f"API Error on charger {r['charger_id']}: {e}")
        aug_data.append({
            'charger_id': r['charger_id'], 'ocm_id': np.nan, 'ocm_operator': 'Unknown', 
            'usage_cost': np.nan, 'ocm_plug_count': 0, 'ocm_plug_types': 'Unknown', 
            'match_method': 'API Error'
        })
        
    time.sleep(0.05)

aug_dataf = pd.DataFrame(aug_data)
final_dataf = dc_chargers.merge(aug_dataf, on='charger_id', how='left')

match_rate = (final_dataf['ocm_id'].notna().sum() / len(final_dataf)) * 100

print("-" * 40)
print(f"Augmentation Complete!")
print(f"Total DC Chargers Evaluated: {len(final_dataf)}")
print(f"Successful OCM Matches: {final_dataf['ocm_id'].notna().sum()}")
print(f"Match Rate: {match_rate:.1f}%")

final_dataf.to_csv('../data/processed/dc_chargers_augmented.csv', index=False)
display(final_dataf[['charger_id', 'operator', 'ocm_operator', 'usage_cost', 'ocm_plug_types', 'match_method']].head(10))

Starting API calls (sit tight and chill) ... it takes some time...


  0%|          | 0/433 [00:00<?, ?it/s]

----------------------------------------
Augmentation Complete!
Total DC Chargers Evaluated: 433
Successful OCM Matches: 355
Match Rate: 82.0%


,charger_id,operator,ocm_operator,usage_cost,ocm_plug_types,match_method
0,2,BP Pulse,BP Pulse (AU),$0.55/kWh,CCS (Type 2),Proximity + Operator Match
1,3,NRMA,NRMA,Currently free but will be fee at some point.,"CCS (Type 2), CHAdeMO",Proximity + Operator Match
2,6,Tesla,Tesla (including non-tesla),0,CCS (Type 2),Proximity + Operator Match
3,7,Evie Networks,Unknown,NaN,Unknown,No Match
4,12,Tesla,Tesla (Tesla-only charging),AUD 0.42/kWh;other tariffs for older cars,CCS (Type 2),Proximity + Operator Match
5,17,Evie Networks,Evie,NaN,"CCS (Type 2), CHAdeMO, CCS (Type 2), CHAdeMO",Proximity + Operator Match
6,20,Chargefox,Unknown,NaN,Unknown,No Match
7,24,Evie Networks,Evie,NaN,"CCS (Type 2), CHAdeMO, CCS (Type 2), CHAdeMO",Proximity + Operator Match
8,28,NRMA,Unknown,NaN,Unknown,No Match
9,36,Jolt,Jolt,"First 7kWh free, then $0.46 per kWh thereafter...","CHAdeMO, CCS (Type 2)",Proximity + Operator Match


In [ ]:
import time
import difflib
import requests
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

ENDPOINT = "https://overpass-api.de/api/interpreter"
HEADERS = {"User-Agent": "COMP5339-Role3-Analysis/1.0 (Atharva; mailto:agad0752@uni.sydney.edu.au)"}

DISTANCE_KM = 3.0

# Pull every OSM charging station in one bounding-box query
pad_deg = DISTANCE_KM / 111.0 + 0.02  # km to degrees, plus small buffer
south = dc_chargers['latitude'].min() - pad_deg
north = dc_chargers['latitude'].max() + pad_deg
west = dc_chargers['longitude'].min() - pad_deg
east = dc_chargers['longitude'].max() + pad_deg

query = f"""
[out:json][timeout:180];
node["amenity"="charging_station"]({south},{west},{north},{east});
out;
"""

print("Fetching all OSM charging stations in bounding box...")
resp = requests.get(ENDPOINT, headers=HEADERS, params={'data': query}, timeout=180)
resp.raise_for_status()
osm_pois = resp.json().get('elements', [])
print(f"Pulled {len(osm_pois)} OSM charging stations")

# Build a KD-tree over OSM points for fast nearest-neighbor search
osm_coords = np.array([[p['lat'], p['lon']] for p in osm_pois])
tree = cKDTree(osm_coords)

deg_radius = DISTANCE_KM / 111.0  # approx conversion for query radius


def operator_match(tfnsw_op: str, osm_operator: str, threshold: float = 0.6) -> bool:
    a, b = tfnsw_op.lower().strip(), osm_operator.lower().strip()
    if not a or not b or b == "unknown":
        return False
    if len(a) < 3 or len(b) < 3:
        return a == b
    return difflib.SequenceMatcher(None, a, b).ratio() >= threshold


# ---- 3. Match each charger locally (no HTTP calls in this loop)
aug_data = []

for i, r in dc_chargers.iterrows():
    idxs = tree.query_ball_point([r['latitude'], r['longitude']], r=deg_radius)

    if idxs:
        candidates = [osm_pois[j] for j in idxs]
        # sort by actual squared distance (degree-approx, consistent with your OCM logic)
        candidates.sort(key=lambda p: (p['lat'] - r['latitude'])**2 + (p['lon'] - r['longitude'])**2)

        best_poi = candidates[0]
        match_method = 'Proximity Only (Closest)'

        tfnsw_op = str(r['operator'])
        for poi in candidates:
            tags = poi.get('tags', {})
            osm_operator = tags.get('operator', tags.get('brand', tags.get('name', 'Unknown')))
            if operator_match(tfnsw_op, osm_operator):
                best_poi = poi
                match_method = 'Proximity + Operator Match'
                break

        final_tags = best_poi.get('tags', {})
        final_osm_operator = final_tags.get('operator', final_tags.get('brand', 'Unknown'))
        capacity = final_tags.get('capacity', np.nan)

        aug_data.append({
            'charger_id': r['charger_id'],
            'osm_id': best_poi.get('id'),
            'osm_operator': final_osm_operator,
            'osm_capacity': capacity,
            'osm_network': final_tags.get('network', 'Unknown'),
            'match_method': match_method
        })
    else:
        aug_data.append({
            'charger_id': r['charger_id'], 'osm_id': np.nan, 'osm_operator': 'Unknown',
            'osm_capacity': np.nan, 'osm_network': 'Unknown', 'match_method': 'No Match'
        })

aug_dataf = pd.DataFrame(aug_data)
final_dataf = dc_chargers.merge(aug_dataf, on='charger_id', how='left')

match_rate = (final_dataf['osm_id'].notna().sum() / len(final_dataf)) * 100

print("-" * 40)
print("Augmentation Complete!")
print(f"Total DC Chargers Evaluated: {len(final_dataf)}")
print(f"Successful OSM Matches: {final_dataf['osm_id'].notna().sum()}")
print(f"Match Rate: {match_rate:.1f}%")

final_dataf.to_csv('../data/processed/dc_chargers_osm_augmented.csv', index=False)
display(final_dataf[['charger_id', 'operator', 'osm_operator', 'osm_network', 'match_method']].head(10))

Fetching all OSM charging stations in bounding box...


HTTPError: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter?data=%0A%5Bout%3Ajson%5D%5Btimeout%3A180%5D%3B%0Anode%5B%22amenity%22%3D%22charging_station%22%5D%28-36.74428632702703%2C141.413076772973%2C-28.126762802972973%2C153.662901927027%29%3B%0Aout%3B%0A

In [ ]:
df['operator'].unique()


<StringArray>
[                        'EVUp',                     'BP Pulse',
                         'NRMA',                    'Chargefox',
                        'Tesla',                'Evie Networks',
                     'AXCharge',                       'Everty',
                     'Exploren',                   'Chargestar',
                         'Jolt',                          'EVX',
                   'Saascharge',                        'Ampol',
                  'ChargePoint',                        'Engie',
                'Non-networked',                       'Wevolt',
                    'ChargeHub',                'EVE Australia',
                  'Viva Energy',       'Porsche Smart Mobility',
                      'Gentari',                     'EV Meter',
                          'BMW',                '360 EV Charge',
 'Porsche Destination Charging',                  'Fast Cities',
                'Energy Austra',                'PLUS ES Manag',
           

In [ ]:
df['operator'].str.len().describe()

count    1958.000000
mean        7.997957
std         3.511714
min         3.000000
25%         5.000000
50%         8.000000
75%         9.000000
max        28.000000
Name: operator, dtype: float64

In [ ]:
df[df['operator'].str.len() == 13]['operator'].unique()

<StringArray>
['Evie Networks', 'Non-networked', 'EVE Australia', '360 EV Charge',
 'Energy Austra', 'PLUS ES Manag', 'University of']
Length: 7, dtype: str

In [ ]:
import json

API_KEY= "145e14a0-0fb0-4e21-8e76-e3d73ee6e696"
ENDPOINT = "https://api.openchargemap.io/v3/poi" 
HEADERS = {"X-API-Key": API_KEY, "User-Agent": "Just-A-Mate-Fetching-EV-Stats/1.0"}

def fetch_raw_ocm(lat, lon, compact_mode):
    params ={
            "latitude": lat,
            "longitude": lon,
            "distance": 3,
            "distanceunit": "KM",
            "maxresults": 2,
            "compact": compact_mode,
            "verbose": False
        }
    try:
        response = requests.get(ENDPOINT, headers=HEADERS, params=params, timeout=10)
        response.raise_for_status()
        return response.json()
    except requests.RequestException as e:
        print(f"API request failed for charger {lat}, {lon}: {e}")
    except ValueError as e:
        print(f"Invalid JSON response for charger {lat}, {lon}: {e}")

    return None
lati = -33.811004
longi = 150.849597
raw_compact = fetch_raw_ocm(lati, longi, True)
raw_full = fetch_raw_ocm(lati, longi, False)

print("COMPACT:")
print(json.dumps(raw_compact, indent=2))
print("FULL:")
print(json.dumps(raw_full, indent=2))
    

COMPACT:
[
  {
    "IsRecentlyVerified": false,
    "DateLastVerified": "2023-08-09T12:21:00Z",
    "ID": 272521,
    "UUID": "99D35B44-B880-47A5-A21D-B10630EDF71C",
    "DataProviderID": 1,
    "OperatorID": 3659,
    "UsageTypeID": 4,
    "UsageCost": "$0.55/kWh",
    "AddressInfo": {
      "ID": 272905,
      "Title": "BP Pulse Eastern Creek",
      "AddressLine1": "BP Truckstop",
      "AddressLine2": "Wallgrove Road & Old Wallgrove Road",
      "Town": "Eastern Creek",
      "StateOrProvince": "New South Wales",
      "Postcode": "2766",
      "CountryID": 18,
      "Latitude": -33.811041669720204,
      "Longitude": 150.85030018744214,
      "AccessComments": "Pass by the car bowsers to the left of the store - charger is approximately just across the driveway at the rear corner of the store",
      "RelatedURL": "https://www.bp.com/en_au/australia/home/products-services/bppulse.html?gclid=CjwKCAjwwb6lBhBJEiwAbuVUShyWLh-WRk2QR_7b7HqMryRBvEu01i9OAzPc3woV7dXV2WJzWxtG6xoCZWQQAvD_BwE"

In [ ]:
test_row = dc_chargers.iloc[0:6]
test_row